## Implementing the BPE algorithm
The aim of this exercise is to implement the BPE tokenization algorithm. As a reminder, the principle consists in gathering the words or “tokens” that appear the most times in succession.

For example, if we consider the corpus containing the words in the following table (with the number of occurrences of each word):

| words | occurrence |
|------|-----------|
| voting | 2 |
| vote | 3 |
| slow | 1 |
| slowly | 2 |


And if the initial “tokens” are the letters of the alphabet, then the prefix “vo” will initially be added to the list of sub-words (tokens), cause the bigram "v" "o" occurs 5 times (2 + 3).

The steps involved in implementing the BPE algorithm are as follows:
1. Download a text corpus (here a wikipedia page)
2. Cut the text into words (using the “space” and “ponctuation” characters) and count the number of occurrences of each word.
3. Initialize the word dictionary with the initial tokens (letters of the alphabet)
4. Run BPE algorithm (learn vocabulary)
5. Test token decomposition on selected sentences (apply learned rules)


In the documents you will observe a lot of TODO in the code, replace it by your code.


### The class we will complete


At the end we will create an object (python) class as it follows :

```python
class Tokenizer:
    ''' 
        A class for our Tokenizer

        Methods
        -------
        fit(text_corpus: str) : None
            Fit the tokenizer based on the input text corpus 
        tokenize(text: str): List[str]
            Tokenize a text and return the list of token ids
        detokenize(tokens: List[str]): str
            From a list of token ids return the corresponding text
    '''

    def __init__(self, vocabulary_size: int = 500):
        ''' 
            Parameters
            ----------
            vocabulary_size : int
                The expected size of the vocabulary

        '''
        super().__init__()
        self.vs = vocabulary_size

    def fit(self, text_corpus: str):
        ''' Train the tokenizer on the provided text.

            Parameters
            ----------
            text_corpus : str
                The text used to train the model
        '''
        raise NotImplementedError


    def tokenize(self, text: str) -> List[str]:
        ''' Tokenize a text.

            Parameters
            ----------
            text : str
                The text to tokenize

            Returns
            -------
            List[str]
                The list of tokens
        '''
        raise NotImplementedError

    def detokenize(self, tokens : List[int]) -> str:
        ''' Reverse the tokenization.

            Parameters
            ----------
            tokens : List[str]
                A list of token ids

            Returns
            -------
            str
                The text corresponding to tokens
        '''
        raise NotImplementedError

```

## Requirement

**For this lab you only need the python standard library :D**

In [1]:
import re # the regex library
import json # read export json format

from typing import List # to specify the types in function def
from collections import Counter # a tools to counts unique occurences

from urllib.request import urlopen, Request

### Step 1: Download a corpus

For this lab we will consider a wikipedia page in french [Grèce antique](https://fr.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&explaintext&redirects=1&titles=Gr%C3%A8ce_antique), but you are free to choose any content you want !

In [2]:
url_request  = 'https://fr.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&explaintext&redirects=1&titles=Gr%C3%A8ce_antique'
wikipedia_request = Request(url_request)
wikipedia_request.add_header("User-Agent", "Course (thomas.gerald@lisn.fr)")
raw_page = urlopen(wikipedia_request)
json_page = json.load(raw_page)

In [3]:
corpus = list(json_page['query']['pages'].values())[0]['extract']

print(f"Corpus length: {len(corpus)} characters")
print()
print(corpus[:500])


Corpus length: 225058 characters

La Grèce antique est une civilisation de l'Antiquité des peuples de langue et de culture grecques développée en Grèce et dans la partie occidentale de l'Asie Mineure, puis, à la suite de plusieurs phases d'expansion, à Chypre, sur le pourtour de la mer Noire, en Sicile, en Italie du sud, en Cyrénaïque, en Égypte, au Levant méridional, en Syrie, constituant des points d'implantation jusque dans les actuelles Espagne et France à l'ouest, et jusqu'au territoire de l’actuel Afghanistan (Bactriane) à


### Step 2: Splitting text into words (or sequence of character no containing space)

To split the text into words, we'll use the following regex ```r'(\b[^\s]+\b)'```. To count words, we'll use python's Counter object. 
1. Store each word and its number of occurrences in **count_words**.
2. Give the 10 most frequent words (you'll store them in most_commons_words).

In [4]:
word_regex = re.compile(r'(\b[^\s]+\b)')
words = word_regex.findall(corpus)

# Count how many times each word appears in the corpus.
count_words = Counter(words)

# Keep the 10 most frequent words
most_commons_words = count_words.most_common(10)

### Step 3: Initialize word dictionary with initial tokens (letters of the alphabet)

Create the initial vocabulary in the vocab variable. How many initial tokens do you have?

In [5]:
# The initial vocabulary is simply every distinct character that appears in the corpus
vocab = sorted({char for word in count_words.keys() for char in word})
print(f"Number of initial tokens: {len(vocab)}")
print(vocab)


Number of initial tokens: 105
["'", '(', ')', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '²', 'À', 'Â', 'É', 'à', 'â', 'ç', 'è', 'é', 'ê', 'î', 'ï', 'ó', 'ô', 'ù', 'û', 'ü', 'Œ', 'œ', 'Α', 'Γ', 'Ε', 'Ι', 'Ο', 'Υ', 'Φ', 'Χ', 'Ψ', 'Ω', 'α', 'ι', 'κ', 'ρ', 'ς', 'ό', '’']


**How many initial tokens?** With this corpus, `vocab` above contains every distinct character used in the text (letters, accented letters like `è`/`é`/`à`, punctuation such as `,` `.` `'`, and digits, greek letters like `Ψ`/`Ω`/`ρ`). The exact count is printed by the cell above (105 tokens).

### Step 4: Learning the tokenizer
To learn the tokenizer we need several functions:
1. A function to calculate the frequency of each token pair.
2. A function to merge a pair

Several variables will be required:
1. **vocab** containing current vocabulary
2. **merge_rules** containing all the merge rules (a dictionary containing as key a pair of tokens to merge and the result of the token merge). For example: {('e', 's'), 'es', ('en', 't') :'ent'}.
3. **splits** A dictionary containing the current breakdown of the corpus, with the word as key and the list of “tokens” as value.


In [6]:
# Each word is initially split into a list of individual characters.
splits = {}

for word in count_words.keys():
    chars = []

    for char in word:
        chars.append(char)

    splits[word] = chars

print(list(splits.items())[:10]) # Print ONLY the first 10 words and their splits

[('La', ['L', 'a']), ('Grèce', ['G', 'r', 'è', 'c', 'e']), ('antique', ['a', 'n', 't', 'i', 'q', 'u', 'e']), ('est', ['e', 's', 't']), ('une', ['u', 'n', 'e']), ('civilisation', ['c', 'i', 'v', 'i', 'l', 'i', 's', 'a', 't', 'i', 'o', 'n']), ('de', ['d', 'e']), ("l'Antiquité", ['l', "'", 'A', 'n', 't', 'i', 'q', 'u', 'i', 't', 'é']), ('des', ['d', 'e', 's']), ('peuples', ['p', 'e', 'u', 'p', 'l', 'e', 's'])]


#### Compute the frequency of token pairs
Create a function **compute_pair_freqs**, which, given the words broken down into tokens (splits dictionary) and the frequency of the words, returns the frequency of each pair of tokens (note only successive sub-words).

In [7]:
def compute_pair_freqs(splits, word_freqs):
    pair_freqs = {}
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] = pair_freqs.get(pair, 0) + freq
    return pair_freqs


In [8]:
pair_freqs = compute_pair_freqs(splits, count_words)
{k: pair_freqs[k] for k  in list(pair_freqs.keys())[:5]}

{('L', 'a'): 165,
 ('G', 'r'): 256,
 ('r', 'è'): 244,
 ('è', 'c'): 270,
 ('c', 'e'): 1162}

#### Find the most frequent pair and merge a pair
1. Create a function **most_frequent(pair_freqs)** returning the most frequent pair of tokens.
2. Create a **merge_pair()** function which, given a pair, returns the new splits of the corpus.

In [9]:
def most_frequent(pair_freqs):
    if not pair_freqs:
        return None
    return max(pair_freqs.items(), key=lambda item: item[1])

most_frequent(pair_freqs)

(('e', 's'), 5657)

In [10]:
def merge_pair(a : str, b: str, splits):
    '''
        splits : the dataset of words tokenized
        a : the first token  
        b : the second tokens

        return : splits with a,b tokens merged
    '''
    for word in splits.keys():
        tokens = splits[word]
        merged = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                merged.append(a + b)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        splits[word] = merged
    return splits

In [11]:
new_splits = merge_pair(*most_frequent(pair_freqs)[0], splits)
print(new_splits['grecques'])

['g', 'r', 'e', 'c', 'q', 'u', 'es']


#### Apply the algorithm until the desired vocabulary size is reached.
Create a BPE object that takes as arguments a corpus, a vocabulary size and train the BPE algorithm. The algorithm stores the final vocabulary in the **vocabulary** attribute and the merge rules in **merge_rules**.
For merge_rules, here's an example of its contents:
```
{('e', 's'): 'es',
 ('n', 't'): 'nt',
 ('q', 'u'): 'qu',
 ('r', 'e'): 're',
 ('o', 'n'): 'on',
 ('d', 'e'): 'de',
 ('l', 'e'): 'le',
 ('t', 'i'): 'ti',
 ('l', 'a'): 'la',
 ('i', 's'): 'is',
 ('e', 'nt'): 'ent', ...
 }
```

In [12]:
class BPE:
    def __init__(self, corpus, vocabulary_size=500):
        super().__init__()
        self.word_regex = re.compile(r'(\b[^\s]+\b)')
        words = self.word_regex.findall(corpus)

        # counting words
        count_words = Counter(words)
        # create initial vocab
        self.vocab = list({char for word in count_words.keys() for char in word })
        self.vocab.sort()
        # create the initial split
        splits = {word: [c for c in word] for word in count_words.keys()}
        # initialise merge_rules
        self.merge_rules = {}

        while len(self.vocab) < vocabulary_size:
            pair_freqs = compute_pair_freqs(splits, count_words)
            if not pair_freqs:
                break

            (best_pair, _), = sorted(pair_freqs.items(), key=lambda x: (-x[1], x[0]))[:1]
            if best_pair in self.merge_rules:
                break

            merged_token = ''.join(best_pair)
            self.merge_rules[best_pair] = merged_token
            self.vocab.append(merged_token)
            self.vocab = sorted(set(self.vocab))
            splits = merge_pair(*best_pair, splits)

    def tokenize(self, text):
        words = self.word_regex.findall(text)
        splits = [[l for l in word] for word in words]
        for pair, merge in self.merge_rules.items():
            for idx, split in enumerate(splits):
                i = 0
                while i < len(split) - 1:
                    if split[i] == pair[0] and split[i + 1] == pair[1]:
                        split = split[:i] + [merge] + split[i + 2 :]
                    else:
                        i += 1
                splits[idx] = split

        return sum(splits, [])

In [13]:
my_bpe = BPE(corpus)

In [14]:
texte = '''culture grecques développée en Grèce '''
my_bpe.tokenize(texte)[:12]

['culture', 'grecques', 'dévelop', 'p', 'ée', 'en', 'Grèce']

#### Test by modifying parameters or corpus
Test the algorithm with different hyper-parameters or data

In [15]:
# Try a larger vocabulary size on the same corpus, and see how the tokenization
# of the same test sentence changes as more merges are learned.
my_bpe_small = BPE(corpus, vocabulary_size=300)
my_bpe_medium = BPE(corpus, vocabulary_size=400)

texte = '''culture grecques développée en Grèce '''
print("vocabulary_size=300:", my_bpe_small.tokenize(texte))
print("vocabulary_size=400:", my_bpe_medium.tokenize(texte))
print("vocabulary_size=500:", my_bpe.tokenize(texte))


vocabulary_size=300: ['culture', 'grec', 'ques', 'dé', 'v', 'el', 'op', 'p', 'ée', 'en', 'Grèce']
vocabulary_size=400: ['culture', 'grec', 'ques', 'dévelop', 'p', 'ée', 'en', 'Grèce']
vocabulary_size=500: ['culture', 'grecques', 'dévelop', 'p', 'ée', 'en', 'Grèce']


**Comment:** As `vocabulary_size` increases, more merges are learned, so words are cut into fewer, longer sub-word pieces (in the limit, with a large enough vocabulary relative to the corpus, entire frequent words become single tokens). With a small vocabulary (300) words stay split into short, very generic fragments, with a larger vocabulary (500) common words like "Grèce" or "culture" are much more likely to already be single merged tokens, since our corpus is short and its vocabulary of unique words is limited, and the medium-sized vocabulary (400) falls somewhere in between. The ceiling on how large a *useful* vocabulary can be is set by how much repetition exists in the training corpus, not just by the `vocabulary_size` parameter itself.

## Using sentencepiece
We're now going to use the `sentencepiece` library, which can be installed (if not already installed) with pip : 

`! pip install sentencepiece`

To train the tokenizer, we'll use the `train` function of `SentencePieceTrainer`. 

In [16]:
import sentencepiece as spm

with open("test-cours.txt", "w", encoding="utf-8") as f:
    f.write(corpus)
spm.SentencePieceTrainer.train(input="test-cours.txt", model_type='BPE',  model_prefix='m', vocab_size=500)


True

In [17]:
# Inspect the vocabulary learned by sentencepiece
with open("m.vocab", encoding="utf-8") as f:
    sp_vocab_lines = [line.rstrip("\n") for line in f]

print(f"sentencepiece vocab size: {len(sp_vocab_lines)}")
print("First 20 entries (token, log-probability score):")
for line in sp_vocab_lines[:20]:
    print(" ", line)

sp_tokens = {line.split("\t")[0] for line in sp_vocab_lines}
our_tokens = set(my_bpe.vocab)
common = sp_tokens & our_tokens
print()
print(f"Our vocab size: {len(our_tokens)}")
print(f"Tokens shared between the two vocabularies: {len(common)}")


sentencepiece vocab size: 500
First 20 entries (token, log-probability score):
  <unk>	0
  <s>	0
  </s>	0
  es	-0
  ▁d	-1
  ▁l	-2
  nt	-3
  ▁p	-4
  qu	-5
  re	-6
  ▁c	-7
  on	-8
  ▁s	-9
  ▁e	-10
  ▁a	-11
  ▁de	-12
  ti	-13
  is	-14
  ent	-15
  ur	-16

Our vocab size: 500
Tokens shared between the two vocabularies: 237


Looking at the "m.vocab" file, what are the differences with the vocabulary learned with your implementation?

**Answer:** A few systematic differences show up when comparing the two vocabularies:

- **Word-boundary handling:** `sentencepiece` (in its default settings) treats the *space* character itself as part of the token stream (it replaces spaces with the meta-symbol `▁` before applying BPE), so a token like `▁le` means "a space followed by 'le'", i.e. "le" at the *start* of a word. Our implementation instead splits the corpus into words first with a regex and runs BPE independently *within* each word. But  it has **no explicit boundary marker at all** (no `▁`, no `</w>`): `splits` simply starts as the bare characters of each word. This means a merged token like `es` in our vocabulary could, for example, equally be the end of "les" or the start of "estudiantin". So, the information about *where in the word* a token sits is lost, whereas sentencepiece's `▁` always tells you a token began a word.

- **Special/control tokens:** `m.vocab` always reserves a few IDs for special tokens (`<unk>`, `<s>`, `</s>`) with very negative scores; our vocabulary has no equivalent because we never designed for out-of-vocabulary handling or sequence boundaries.
- **Merge order / tie-breaking:** Both implementations are greedy and merge the globally most frequent pair at each step **(as per the `sentencepiece` documentation)**, that's why they agree on the most frequent merges (like very common bigrams). Our implementation explicitly breaks ties between equally-frequent pairs by lexicographic order on the pair, which is deterministic but is not necessarily the same rule sentencepiece uses internally (its suffix-array-based implementation has its own undocumented tie-breaking behaviour), so the two can still diverge on pairs with equal frequency. They also diverge because of `sentencepiece`'s pre-tokenization/normalization choices (for example how it treats punctuation and digits), which differ from our `\b[^\s]+\b` regex.

## Detokenization

Propose and implement a methods to detokenize encoded sentence (you should want to extend your vocabulary)

**Proposal.** `tokenize()` as originally written throws away all information about where one word ends and the next begins: it just concatenates the per-word token lists into one flat list (`sum(splits, [])`), so `''.join(tokens)` would glue every word in the sentence together with no spaces, which is not reversible.

The standard fix (used by real BPE tokenizers such as GPT-2's) is to **extend the vocabulary with an explicit boundary marker** that becomes part of the tokens themselves, so the boundary information survives merges and survives being flattened into one list. We do this in the new `ExtendedBPE` class by appending an end-of-word marker `</w>` as an extra "character" to every word before the very first merge. Because merges only ever combine two *adjacent* tokens, and `</w>` starts out at the end of a word, it can only ever be merged into the token immediately before it. So `</w>` (or a merged token ending in `</w>`) always marks the end of a word, never the middle. `detokenize()` then simply concatenates all tokens and turns every `</w>` marker back into a plain space.

In [18]:
class ExtendedBPE:

    EOW = '</w>'

    def __init__(self, corpus, vocabulary_size=500):
        super().__init__()
        self.word_regex = re.compile(r'(\b[^\s]+\b)')
        words = self.word_regex.findall(corpus)

        # counting words
        count_words = Counter(words)
        # create initial vocab (characters + the end-of-word marker)
        self.vocab = list({char for word in count_words.keys() for char in word})
        self.vocab.append(self.EOW)
        self.vocab.sort()
        # create the initial split: characters of the word, plus the EOW marker
        splits = {word: [c for c in word] + [self.EOW] for word in count_words.keys()}
        # initialise merge_rules
        self.merge_rules = {}
        while len(self.vocab) < vocabulary_size:
            pair_freqs = compute_pair_freqs(splits, count_words)
            if not pair_freqs:
                # no more pairs to merge (every word is now a single token)
                break
            best_pair, best_freq = most_frequent(pair_freqs)
            splits = merge_pair(*best_pair, splits)
            new_token = best_pair[0] + best_pair[1]
            self.merge_rules[best_pair] = new_token
            self.vocab.append(new_token)

    def tokenize(self, text):
        words = self.word_regex.findall(text)
        splits = [[l for l in word] + [self.EOW] for word in words]
        for pair, merge in self.merge_rules.items():
            for idx, split in enumerate(splits):
                i = 0
                while i < len(split) - 1:
                    if split[i] == pair[0] and split[i + 1] == pair[1]:
                        split = split[:i] + [merge] + split[i + 2 :]
                    else:
                        i += 1
                splits[idx] = split

        return sum(splits, [])

    def detokenize(self, tokens):
        ''' Reverse the tokenization.

            Parameters
            ----------
            tokens : List[str]
                A list of token ids (as returned by `tokenize`)

            Returns
            -------
            str
                The text corresponding to tokens
        '''
        # Every token that ends a word contains the EOW marker as a suffix
        # (merges can only ever attach it to the token that precedes it, never
        # move it to the middle of a token). So concatenating every token and
        # turning each EOW marker into a plain space reconstructs the original
        # (whitespace-normalized) text.
        text = ''.join(tokens)
        text = text.replace(self.EOW, ' ')
        return text.strip()


In [19]:
# Round-trip demonstration: tokenize, then detokenize, and check we recover the original text.
sentence = "La culture grecque antique a influencé toute la civilisation occidentale."

my_extended_bpe = ExtendedBPE(corpus)

tokens = my_extended_bpe.tokenize(sentence)
print("Tokens:", tokens)

reconstructed = my_extended_bpe.detokenize(tokens)
print("Reconstructed:", repr(reconstructed))
print("Original (word-regex normalized):", repr(' '.join(my_extended_bpe.word_regex.findall(sentence))))
print("Round-trip successful:", reconstructed == ' '.join(my_extended_bpe.word_regex.findall(sentence)))


Tokens: ['La</w>', 'culture</w>', 'grecque</w>', 'antique</w>', 'a</w>', 'in', 'f', 'lu', 'en', 'c', 'é</w>', 'tou', 'te</w>', 'la</w>', 'civil', 'isation</w>', 'oc', 'ci', 'd', 'enta', 'le</w>']
Reconstructed: 'La culture grecque antique a influencé toute la civilisation occidentale'
Original (word-regex normalized): 'La culture grecque antique a influencé toute la civilisation occidentale'
Round-trip successful: True


The round trip is exact up to whitespace normalization: since `word_regex` only ever splits on whitespace, any run of whitespace in the input becomes a single space in the reconstructed text (so `tokenize` + `detokenize` recovers the original sentence exactly whenever the sentence used single spaces between words, which is the common case).